Exercise 1 - Understanding the problem and Data Collection

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("diabetes_prediction_dataset.csv")

print(df.shape)
display(df.head())
print(df.dtypes)
print("Missing per column:")
display(df.isna().sum().sort_values(ascending=False))

assert 'diabetes' in df.columns, "Expected a 'diabetes' target column"
print(df['diabetes'].value_counts())

X = df.drop(columns=['diabetes'])
y = df['diabetes']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

Exercise 2 - Model picking and standardization

Logistic Regression is perfectly suited for this binary classification task because it maps features linearly to a log-odds scale, outputting well-calibrated probabilities between 0 and 1 via the sigmoid function. Its coefficients offer direct interpretability, allowing us to understand how each medical feature impacts the risk of diabetes. Standardizing numerical features using StandardScaler is crucial here because it brings all variables to the same scale, ensuring stable numerical convergence during optimization and preventing features with larger scales (like blood glucose level) from dominating the model gradient updates.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first'), cat_cols)
])

print("Categorical:", cat_cols)
print("Numeric:", num_cols)

Exercise 3 - Model training

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

clf = Pipeline([
    ('preprocessor', preprocess),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

clf.fit(X_train, y_train)

Exercise 4 - Evaluation metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", round(acc, 4))
print("Precision:", round(prec, 4))
print("Recall:", round(rec, 4))
print("F1:", round(f1, 4))

plt.figure(figsize=(6, 4))
plt.bar(['accuracy','precision','recall','f1'], [acc, prec, rec, f1], color=['blue', 'orange', 'green', 'red'])
plt.title('Metrics on test')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.show()

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Diabetes', 'Diabetes'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion matrix')
plt.show()

Precision vs. Recall Commentary:

In a medical context like diabetes diagnosis, a balance between precision and recall must favor recall to avoid missing true positive cases (false negatives can lead to undiagnosed patients). While a high accuracy might look impressive, the high precision means that when the model predicts diabetes, it is highly likely correct. However, if the recall is lower, it means some diabetic patients are left undetected, showcasing why we optimize the decision threshold using the F1-score to strike a balance.

Exercise 5 - Visualizing the performance of our model

In [ ]:
import numpy as np

feat_x = 'HbA1c_level' if 'HbA1c_level' in X.columns else X.select_dtypes(include=['int64','float64']).columns[0]
feat_y = 'blood_glucose_level' if 'blood_glucose_level' in X.columns else X.select_dtypes(include=['int64','float64']).columns[1]

X2_train = X_train[[feat_x, feat_y]].copy()
X2_test = X_test[[feat_x, feat_y]].copy()

pipe2 = Pipeline([
    ('pre', ColumnTransformer([('num', StandardScaler(), [0,1])], remainder='drop')),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
pipe2.fit(X2_train.values, y_train)

# Meshgrid generation
x_min, x_max = X2_train[feat_x].min() - 1, X2_train[feat_x].max() + 1
y_min, y_max = X2_train[feat_y].min() - 1, X2_train[feat_y].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
probs = pipe2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1].reshape(xx.shape)

plt.figure(figsize=(6,5))
cs = plt.contour(xx, yy, probs, levels=[0.5], colors='red', linewidths=2)
plt.clabel(cs, inline=True, fmt={0.5:'P=0.5'})
plt.scatter(X2_test[feat_x], X2_test[feat_y], c=y_test, alpha=0.5, cmap='coolwarm', edgecolors='k', s=20)
plt.xlabel(feat_x); plt.ylabel(feat_y)

acc2 = accuracy_score(y_test, pipe2.predict(X2_test.values))
plt.title(f'Decision boundary on 2 features - test accuracy {acc2:.3f}')
plt.show()

Exercise 6 - ROC curve

In [ ]:
from sklearn import metrics

y_proba = clf.predict_proba(X_test)[:,1]
fpr, tpr, _ = metrics.roc_curve(y_test, y_proba)
auc = metrics.roc_auc_score(y_test, y_proba)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC={auc:.3f}')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.ylabel('True Positive Rate (Recall)')
plt.xlabel('False Positive Rate')
plt.legend(loc=4)
plt.title('ROC curve')
plt.show()

ROC & AUC Interpretation:

The ROC curve displays the trade-off between the True Positive Rate (Sensitivity) and False Positive Rate across all possible classification thresholds. An AUC score close to 1.0 indicates excellent discriminative power, proving that the model is highly capable of distinguishing between diabetic individuals and healthy individuals regardless of the baseline class imbalance.